# Extended Feature Set — features_9

Builds on top of `features_9` (147 features = features_6 + momentum/vol/calendar/peer returns) by appending deeper cross-pair correlation features:

1. **Rolling correlation** — 24H, 72H, 168H between pair and each peer
2. **Beta** — rolling regression of pair return on peer return (shared vs idiosyncratic)
3. **Relative strength** — pair return minus peer return (isolates currency component)
4. **Currency strength index** — implied strength of each of the 8 currencies across all pairs
5. **Correlation regime** — short-term vs long-term correlation (detects relationship breakdown)

**Input:** `backend/data/features_9/{pair}_features.parquet`
**Output:** `backend/data/features_9/{pair}_features.parquet` (overwrite)

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

FEATURES_9_DIR = Path('../backend/data/features_9')  # input AND output
RAW_DIR        = Path('../backend/data/raw')

PAIRS = [
    'EURUSD', 'GBPUSD', 'USDJPY', 'USDCHF', 'AUDUSD', 'USDCAD', 'NZDUSD',
    'EURJPY', 'GBPJPY', 'EURGBP', 'EURAUD', 'AUDJPY', 'CADJPY', 'CHFJPY', 'AUDNZD'
]

# Cross-pair peers: pairs sharing base or quote currency
CROSS_PEERS = {
    'EURUSD': ['GBPUSD', 'AUDUSD', 'EURGBP', 'EURJPY'],
    'GBPUSD': ['EURUSD', 'AUDUSD', 'EURGBP', 'GBPJPY'],
    'USDJPY': ['EURJPY', 'GBPJPY', 'AUDJPY', 'CADJPY'],
    'USDCHF': ['EURUSD', 'GBPUSD', 'CHFJPY'],
    'AUDUSD': ['EURUSD', 'GBPUSD', 'NZDUSD', 'AUDNZD', 'AUDJPY'],
    'USDCAD': ['CADJPY', 'EURUSD', 'GBPUSD'],
    'NZDUSD': ['AUDUSD', 'AUDNZD', 'EURUSD'],
    'EURJPY': ['USDJPY', 'GBPJPY', 'AUDJPY', 'EURUSD'],
    'GBPJPY': ['USDJPY', 'EURJPY', 'AUDJPY', 'GBPUSD'],
    'EURGBP': ['EURUSD', 'GBPUSD', 'EURJPY'],
    'EURAUD': ['EURUSD', 'AUDUSD', 'AUDNZD'],
    'AUDJPY': ['USDJPY', 'EURJPY', 'AUDUSD', 'AUDNZD'],
    'CADJPY': ['USDJPY', 'EURJPY', 'USDCAD'],
    'CHFJPY': ['USDJPY', 'EURJPY', 'USDCHF'],
    'AUDNZD': ['AUDUSD', 'NZDUSD', 'AUDJPY'],
}

# Currency composition: for each pair, which currency is base (+1) and quote (-1)
# Used to build currency strength index
CURRENCY_SIGN = {
    'EURUSD': {'EUR': +1, 'USD': -1},
    'GBPUSD': {'GBP': +1, 'USD': -1},
    'USDJPY': {'USD': +1, 'JPY': -1},
    'USDCHF': {'USD': +1, 'CHF': -1},
    'AUDUSD': {'AUD': +1, 'USD': -1},
    'USDCAD': {'USD': +1, 'CAD': -1},
    'NZDUSD': {'NZD': +1, 'USD': -1},
    'EURJPY': {'EUR': +1, 'JPY': -1},
    'GBPJPY': {'GBP': +1, 'JPY': -1},
    'EURGBP': {'EUR': +1, 'GBP': -1},
    'EURAUD': {'EUR': +1, 'AUD': -1},
    'AUDJPY': {'AUD': +1, 'JPY': -1},
    'CADJPY': {'CAD': +1, 'JPY': -1},
    'CHFJPY': {'CHF': +1, 'JPY': -1},
    'AUDNZD': {'AUD': +1, 'NZD': -1},
}

CURRENCIES = ['EUR', 'USD', 'GBP', 'JPY', 'AUD', 'NZD', 'CAD', 'CHF']

print('Setup complete.')
print(f'Input/output directory: {FEATURES_9_DIR}')

## 1. Load 1H Returns for All Pairs

Load raw 1-minute data and resample to 1H close prices and log returns.
These are needed to compute correlation, beta, and currency strength features.

In [ ]:
print('Loading 1H close prices...')
close_1h = {}

for pair in PAIRS:
    yearly = []
    for f in sorted(RAW_DIR.glob(f'{pair}_1min_*.parquet')):
        tmp = pd.read_parquet(f, columns=['datetime', 'close'])
        tmp['datetime'] = pd.to_datetime(tmp['datetime'])
        tmp = tmp.set_index('datetime')
        yearly.append(tmp)
    if not yearly:
        print(f'  WARNING: no raw data for {pair}')
        continue
    df_1m = pd.concat(yearly).sort_index()
    df_1h = df_1m['close'].resample('1h').last().dropna()
    df_1h.index.name = 'datetime'
    close_1h[pair] = df_1h
    print(f'  {pair}: {len(df_1h):,} hours ({df_1h.index.min().date()} to {df_1h.index.max().date()})')

print(f'\nLoaded {len(close_1h)} pairs.')

## 2. Compute Currency Strength Index

For each of the 8 currencies (EUR, USD, GBP, JPY, AUD, NZD, CAD, CHF), compute its
implied strength at each hour by averaging the signed returns of all pairs involving it.

e.g. USD strength = mean(+USDJPY_ret, +USDCHF_ret, +USDCAD_ret, -EURUSD_ret, -GBPUSD_ret, -AUDUSD_ret, -NZDUSD_ret)

In [ ]:
# Build returns matrix: one column per pair, hourly log returns
all_returns = {}
for pair in PAIRS:
    if pair not in close_1h:
        continue
    c = close_1h[pair]
    all_returns[pair] = np.log(c / c.shift(1))

returns_df = pd.DataFrame(all_returns)

# Currency strength index: for each currency, average signed returns across all pairs
print('Computing currency strength index...')
csi = {}
for ccy in CURRENCIES:
    components = []
    for pair, signs in CURRENCY_SIGN.items():
        if ccy in signs and pair in returns_df.columns:
            components.append(signs[ccy] * returns_df[pair])
    if components:
        csi[f'csi_{ccy.lower()}'] = pd.concat(components, axis=1).mean(axis=1)

csi_df = pd.DataFrame(csi)

# Rolling CSI (24H momentum of each currency)
csi_rolling = {}
for col in csi_df.columns:
    ccy = col.replace('csi_', '')
    csi_rolling[f'{col}_24h'] = csi_df[col].rolling(24, min_periods=8).sum()
    csi_rolling[f'{col}_72h'] = csi_df[col].rolling(72, min_periods=24).sum()

csi_rolling_df = pd.DataFrame(csi_rolling)

print(f'CSI columns: {list(csi_df.columns)}')
print(f'CSI rolling columns: {len(csi_rolling_df.columns)}')

## 3. Compute Per-Pair Correlation Features

For each pair and each peer:
- **Rolling correlation** (24H, 72H, 168H)
- **Beta** (rolling OLS: pair_ret = beta * peer_ret)
- **Relative strength** (pair_ret - peer_ret = isolated currency component)
- **Correlation regime** (corr_24H - corr_168H = short vs long-term divergence)

In [ ]:
def compute_correlation_features(pair, returns_df, csi_df, csi_rolling_df):
    if pair not in returns_df.columns:
        return pd.DataFrame()

    r = returns_df[pair]
    feat = pd.DataFrame(index=r.index)

    peers = CROSS_PEERS.get(pair, [])

    for peer in peers:
        if peer not in returns_df.columns:
            continue
        p = returns_df[peer]
        slug = peer.lower()

        # Rolling correlation
        for w, label in [(24, '24h'), (72, '3d'), (168, '1w')]:
            feat[f'corr_{slug}_{label}'] = r.rolling(w, min_periods=w//2).corr(p)

        # Correlation regime: short vs long-term
        feat[f'corr_regime_{slug}'] = (
            feat[f'corr_{slug}_24h'] - feat[f'corr_{slug}_1w']
        )

        # Beta: rolling OLS (pair = beta * peer)
        for w, label in [(24, '24h'), (168, '1w')]:
            cov = r.rolling(w, min_periods=w//2).cov(p)
            var = p.rolling(w, min_periods=w//2).var().clip(lower=1e-12)
            feat[f'beta_{slug}_{label}'] = cov / var

        # Relative strength (isolated currency component)
        feat[f'relstr_{slug}_1h']  = r - p
        feat[f'relstr_{slug}_4h']  = (
            np.log(close_1h[pair]  / close_1h[pair].shift(4)) -
            np.log(close_1h[peer]  / close_1h[peer].shift(4))
        ).reindex(r.index)
        feat[f'relstr_{slug}_24h'] = (
            np.log(close_1h[pair]  / close_1h[pair].shift(24)) -
            np.log(close_1h[peer]  / close_1h[peer].shift(24))
        ).reindex(r.index)

    # Currency strength features for this pair's currencies
    for ccy, sign in CURRENCY_SIGN.get(pair, {}).items():
        col = f'csi_{ccy.lower()}'
        if col in csi_df.columns:
            feat[col]              = csi_df[col].reindex(r.index)
            feat[f'{col}_24h']     = csi_rolling_df[f'{col}_24h'].reindex(r.index)
            feat[f'{col}_72h']     = csi_rolling_df[f'{col}_72h'].reindex(r.index)

    return feat.astype(np.float32)

print('Correlation feature function ready.')

## 4. Merge & Overwrite features_9

In [ ]:
print('Merging correlation features into features_9...')

for pair in PAIRS:
    f9_path = FEATURES_9_DIR / f'{pair}_features.parquet'
    if not f9_path.exists():
        print(f'  SKIP {pair} -- no features_9')
        continue

    df9 = pd.read_parquet(f9_path)

    corr_feat = compute_correlation_features(pair, returns_df, csi_df, csi_rolling_df)
    corr_feat = corr_feat.reindex(df9.index)

    # Only add columns not already present
    existing = set(df9.columns)
    new_cols = [c for c in corr_feat.columns if c not in existing]
    corr_feat = corr_feat[new_cols]

    df9_out = df9.join(corr_feat, how='left')
    df9_out.to_parquet(f9_path)  # overwrite
    print(f'  {pair}: {len(df9.columns)} -> {len(df9_out.columns)} cols ({len(new_cols)} new)')

print(f'\nDone. features_9 overwritten.')

## 5. Summary

In [ ]:
# Sanity check on EURUSD
df_check = pd.read_parquet(FEATURES_9_DIR / 'EURUSD_features.parquet')
f9_orig_cols = 147  # before this notebook run

new_cols = df_check.columns[f9_orig_cols:].tolist()
print(f'EURUSD features_9: {df_check.shape}')
print(f'New correlation features ({len(new_cols)}):')
for c in new_cols[:30]:
    null_pct = df_check[c].isna().mean()
    print(f'  {c:<40} nulls={null_pct:.1%}')
if len(new_cols) > 30:
    print(f'  ... and {len(new_cols)-30} more')

print(f'\nFeature groups summary:')
print(f'  Rolling correlation (corr_*):   {sum(1 for c in new_cols if c.startswith("corr_") and "regime" not in c)}')
print(f'  Correlation regime (corr_reg*): {sum(1 for c in new_cols if "regime" in c)}')
print(f'  Beta (beta_*):                  {sum(1 for c in new_cols if c.startswith("beta_"))}')
print(f'  Relative strength (relstr_*):   {sum(1 for c in new_cols if c.startswith("relstr_"))}')
print(f'  Currency strength (csi_*):      {sum(1 for c in new_cols if c.startswith("csi_"))}')
print(f'\nNext step: retrain notebooks_8/02_model_training.ipynb')